# MuscleMap WB Segmentation — Asian MRI Dataset (Water, Lambda)

Runs **MuscleMap whole-body** segmentation on the `MRI_data_asian` Dixon WATER stacks.

Structure: `MRI_data_asian/MRI_data/{01-25}/{Thigh|Calf}/Water.nii.gz`  
Output: `~/asian_segs_water/{subject}/{region}/Water_dseg.nii.gz` (50 files total)

## 1 — Upload data to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/MRI_data_asian \
  your_machine:~/
```

## 2 — Download results when done
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your_machine:~/asian_segs_water/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/asian_segs_water/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os, shutil

# Search common conda locations on Lambda/Ubuntu
_conda_candidates = [
    shutil.which('conda'),
    os.path.expanduser('~/miniconda3/bin/conda'),
    os.path.expanduser('~/anaconda3/bin/conda'),
    '/opt/conda/bin/conda',
    '/usr/local/conda/bin/conda',
]
CONDA = next((p for p in _conda_candidates if p and os.path.exists(p)), None)

if CONDA is None:
    print('conda not found — installing Miniconda ...')
    subprocess.check_call([
        'bash', '-c',
        'wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh '
        '-O /tmp/miniconda.sh && bash /tmp/miniconda.sh -b -p ~/miniconda3'
    ])
    CONDA = os.path.expanduser('~/miniconda3/bin/conda')

print('conda:', CONDA)

ENV_NAME     = 'musclemap_env'
conda_prefix = os.path.dirname(os.path.dirname(CONDA))
ENV_DIR      = os.path.join(conda_prefix, 'envs', ENV_NAME)
ENV_PY       = os.path.join(ENV_DIR, 'bin', 'python')

if not os.path.exists(ENV_PY):
    print('Creating conda env with Python 3.11 ...')
    subprocess.check_call([
        CONDA, 'create', '-n', ENV_NAME, 'python=3.11', 'pip',
        '-c', 'conda-forge', '--override-channels',
        '-y', '-q',
    ])
    print('Env created.')
else:
    print('Conda env already exists.')

def env_pip(*args):
    subprocess.check_call([ENV_PY, '-m', 'pip'] + list(args))

env_pip('install', '--upgrade', '-q', 'pip')
env_pip('install', '-q', 'git+https://github.com/MuscleMap/MuscleMap.git')

MM_BIN = os.path.join(ENV_DIR, 'bin', 'mm_segment')
print('mm_segment exists:', os.path.exists(MM_BIN))

In [ ]:
import glob, os

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────

DATA_ROOT  = os.path.expanduser('~/MRI_data_asian/MRI_data')
OUTPUT_DIR = os.path.expanduser('~/asian_segs_water')
REGIONS    = ['Thigh', 'Calf']

# Collect (subject, region, water_path) tuples
jobs = []
for subject in sorted(os.listdir(DATA_ROOT)):
    for region in REGIONS:
        water_path = os.path.join(DATA_ROOT, subject, region, 'Water.nii.gz')
        if os.path.exists(water_path):
            jobs.append((subject, region, water_path))

print(f'Found {len(jobs)} Water stacks')
for subj, reg, p in jobs:
    print(f'  {subj}/{reg}  →  {p}')

In [ ]:
# ── Run MuscleMap ─────────────────────────────────────────────────────────────
#
# All inputs are named Water.nii.gz — route each job to its own subdirectory
# so outputs don't overwrite each other:
#   ~/asian_segs_water/{subject}/{region}/Water_dseg.nii.gz
#
# Skip-check uses glob because mm_segment strips extensions differently for
# .nii.gz vs .nii inputs (may produce Water_dseg.nii.gz or Water.nii_dseg.nii.gz).

run_env = os.environ.copy()
run_env.pop('MPLBACKEND', None)   # avoid matplotlib backend conflicts in subprocess

for subject, region, water_path in jobs:
    out_subdir = os.path.join(OUTPUT_DIR, subject, region)

    if glob.glob(os.path.join(out_subdir, '*_dseg*')):
        print(f'Skipping (done): {subject}/{region}')
        continue

    os.makedirs(out_subdir, exist_ok=True)
    print(f'\nProcessing: {subject}/{region}')
    subprocess.check_call([
        MM_BIN,
        '-i', water_path,
        '-r', 'wholebody',
        '-o', out_subdir,
        '-g', 'Y',
    ], env=run_env)
    dseg = glob.glob(os.path.join(out_subdir, '*_dseg*'))
    print(f'  Saved → {dseg}')

print('\nAll done.')

In [ ]:
# ── Sanity check (runs inside the conda env so SimpleITK is available) ────────

results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', '*', '*_dseg*')))
print(f'Output files found: {len(results)} / {len(jobs)}')

if results:
    script = f"""
import SimpleITK as sitk, numpy as np
img = sitk.ReadImage({repr(results[0])})
arr = sitk.GetArrayFromImage(img)
labels = sorted(np.unique(arr[arr > 0]).tolist())
print('Sample :', {repr(results[0])})
print('Shape  :', arr.shape)
print('Labels :', labels)
"""
    out = subprocess.check_output([ENV_PY, '-c', script], env=run_env)
    print(out.decode())